## Voting Classifiers

In [8]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
voting_clf = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(random_state=42)),
        ("rf", RandomForestClassifier(random_state=42)),
        ("svc", SVC(random_state=42)),
    ]
)

# VotingClassifier когда обучаем клонирует каждый estimator и обучает клоны
# оригинал доступен по аттрибуту estimators
# а клон по estimators_
voting_clf.fit(X_train, y_train)

# Вывод score каждого классификатора
for name, clf in voting_clf.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test))

# predict вызывает жесткое голосование
print(voting_clf.predict(X_test[:1]))
# Крч у нас выше вышло 1, так как большинство сказало что 1
# Хотя если посмотреть два предсказало 1, а один из них сказал 0, поэтому ответ 1
print([clf.predict(X_test[:1]) for clf in voting_clf.estimators_])



lr = 0.864
rf = 0.896
svc = 0.896
[1]
[array([1], dtype=int64), array([1], dtype=int64), array([0], dtype=int64)]


In [ ]:
# В этом примере Voting превосходит все отдельные модели на тесте.
print(voting_clf.score(X_test, y_test))
# Механика: он смотрит на ответы 3 классификаторов и берёт класс,
# за который больше голосов (доверяет большинству).
# Это выглядит как компенсация ошибки того, кто не разобрался:
# его голос перекрывают двое других.
# При этом те двое тоже могут ошибаться — Voting не знает, кто прав,
# он просто считает голоса. Но так как модели ошибаются на разных
# точках, большинство чаще оказывается правым.

0.92


In [ ]:
# Soft voting
# - Hard voting: каждая модель говорит класс (0 или 1), считаем голоса.
# - Soft voting: каждая модель говорит вероятности, усредняем их и берём 
# класс с большей средней вероятностью.
voting_clf.voting = "soft"
voting_clf.named_estimators["svc"].probability = True
# SVC(probability=True) нужен потому, что SVC по умолчанию не умеет 
# predict_proba() — он выдаёт только класс.
voting_clf.fit(X_train, y_train)
voting_clf.score(X_test, y_test)
# Ну и дак так выходит что он оказался еще лучше чем прошлый
# Так как в прошлом были просто классы или вроде того

0.92

## Bagging and Pasting